# Four independent disease classifiers

Mentor notebook for MediAI. Each disease is its **own** binary model. There is no combined four-disease network.

**What this notebook actually does (not a sketch):** every training cell below **fits** XGBoost, LightGBM, CatBoost and TabPFN. Optuna trial lines are printed as they run (CV ROC-AUC, F1, accuracy, hyperparameters). The winning algorithm is refit, scored on a frozen holdout, explained with SHAP, and saved.

**Accuracy target:** quoted holdout accuracy must be **≥ 85%**, and that number has to beat the majority-class dummy (otherwise 85% is just class imbalance). Training uses **multiple same-schema datasets per disease**. Trees are **unweighted** so Optuna is not buying recall by spending accuracy.

TabPFN uses the **open TabPFNv2** checkpoint (Prior-Labs/TabPFN-v2-clf). TabPFN 9.x would otherwise default to gated v3.5 weights that need a Prior Labs login; that is not used here. On tables larger than 4000 train rows, TabPFN is fit on a stratified 4000-row cap; the trees still see the full pool.

These are screening-style classifiers on public tables. They are **not** diagnoses and not clinician-signed.

| Disease | Training rows (pooled) | Why this data |
|---|---|---|
| Heart | Indian hospital (n=1000) **plus** UCI Cleveland / Hungary / Switzerland / VA | India-first, same 13 clinical fields as the app; extra sites for pooling |
| Liver | ILPD Andhra Pradesh **plus** UCI HCV **plus** Mayo PBC (UCI 878) | ILPD matches the app form; HCV is donor vs hepatitis; Mayo adds confirmed liver-disease rows on overlapping labs. No extra public ILPD-like labelled table exists (IEEE 51-row CLD is login-walled; NHANES “ever liver condition” dropped ILPD accuracy). |
| Diabetes | Pima 8-lab **plus** Pabna 8-lab (Bangladesh, DOI 10.17632/vxnyysk9vc.3) **plus** NHANES 2011–2023 overlapping labs | Pima matches the app form. Pabna is the only extra public 8-lab table that is not a Pima clone (0 overlapping glucose+age+BMI keys). NHANES supplies thousands of extra labeled adults on glucose/BMI/age/diastolic BP. Negatives are subsampled to ~1.8× positives so prevalence stays ~Pima (≈35%) and 85% cannot be a dummy. |
| Kidney | **Served model:** four labs the form types (creatinine, urea, hemoglobin, BP) on UCI CKD Tamil Nadu + Bangladesh | Do **not** quote the 24-column hospital-table ~100% as the chat score. The app never collects urine microscopy / sodium / WBC. |

**Protocol (identical for every disease):** stratified 80/20 holdout → Optuna on the 80% maximizing 5-fold (3-fold for TabPFN) **ROC-AUC** → among algorithms with CV accuracy ≥ 0.85, pick the best CV AUC (else best CV AUC overall) → score the 20% once at threshold 0.5 → SHAP on the winner. Holdout-by-source is printed so a high pooled number cannot hide a weak site.


In [ ]:
import json, os, sys, time
from pathlib import Path
from IPython.display import display, Image, Markdown
import pandas as pd
import matplotlib
matplotlib.use("Agg")

os.environ.setdefault("TABPFN_MODEL_VERSION", "v2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")

ROOT = Path.cwd()
if not (ROOT / "ml" / "train_compare.py").exists():
    # notebook executed from notebooks/
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("project root:", ROOT)
print("python:", sys.version.split()[0])

from ml.data_prep import (
    feature_target, load_heart, load_liver, load_diabetes, load_kidney, load_diabetes_sylhet,
    KIDNEY_FORM_FEATURES,
)
from ml.run_training import train_heart, train_liver, train_diabetes, train_kidney, train_kidney_form
from ml.train_compare import (
    TREE_TRIALS, TABPFN_TRIALS, TREE_FOLDS, TABPFN_FOLDS,
    ACCURACY_GATE, USE_CLASS_WEIGHTS, TABPFN_MAX_ROWS,
)

print(f"Optuna budget: trees {TREE_TRIALS} trials × {TREE_FOLDS}-fold; TabPFN {TABPFN_TRIALS} × {TABPFN_FOLDS}-fold")
print("TabPFN checkpoint: open v2 (TABPFN_MODEL_VERSION=v2)")
print(f"accuracy gate: {ACCURACY_GATE:.0%}  class_weights={USE_CLASS_WEIGHTS}  TabPFN cap={TABPFN_MAX_ROWS}")
RESULTS = {}


---
## 1. Heart disease

**Task.** Presence vs absence of heart disease (binary). UCI `num>0`; Indian `target`.

**Why these datasets.** The app already collects Cleveland-style fields (age, sex, chest pain, BP, cholesterol, ECG, max HR, ST depression, vessels). A 1000-row Indian hospital table (Mendeley DOI 10.17632/dzz48mvjht.1) uses the same panel (no `thal`). UCI Cleveland / Hungary / Switzerland / VA add extra sites with that schema. Chest-pain codes on the India table are 0–3 and are shifted to UCI’s 1–4 before pooling. The India file is not a Cleveland clone (0 overlapping rows on age+BP+cholesterol+max HR).

**Not a diagnosis.** Angiographic tables from the 1980s plus one Indian hospital dump.


In [ ]:
heart_pool, heart_india = load_heart()
print("pooled rows", len(heart_pool), "by source:\n", heart_pool["source"].value_counts().to_string())
print("class balance (1=disease):")
print(heart_pool.groupby("source")["disease"].mean().round(3).to_string())
display(heart_pool.drop(columns=["source"]).head(8))
print("missing counts:\n", heart_pool.isna().sum().to_string())


The next cell **trains** all four algorithms. Each printed `trial` line is a real cross-validated fit, not a placeholder.

In [ ]:
t0 = time.time()
RESULTS["heart"] = train_heart()
print(f"\nheart wall time: {time.time()-t0:.0f}s")
print("winner:", RESULTS["heart"]["winner"])
display(pd.DataFrame(RESULTS["heart"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["heart"]
print("Why this winner:", h["winner"],
      "had the highest cross-validated ROC-AUC among the four tuned algorithms.")
print("CV:", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout:", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Holdout by hospital source:", json.dumps(h.get("winner_holdout_by_source"), indent=2, default=str)[:1500])
print("Best hyperparameters:", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("\nSHAP: bar = mean |impact|; beeswarm = direction per patient on the train split.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


---
## 2. Liver disease

**Task.** Liver-disease vs not. ILPD `Selector==1`; HCV non-donor vs blood donor (suspect donors dropped).

**Why these datasets.** ILPD (DOI 10.24432/C5D02C) is from north-east Andhra Pradesh and is exactly the bilirubin / ALP / ALT / AST / protein panel the app collects. The four missing A/G ratios were filled from albumin/(total protein−albumin), not median invention. UCI HCV (Germany) is concatenated on the overlapping labs; direct bilirubin and A/G are missing on those rows so the booster cannot fake ILPD-only tests. Mayo PBC (UCI 878, DOI 10.24432/C5R02G) adds confirmed liver-disease rows (Age, Sex, bilirubin, ALP, AST, albumin). `Status` / `Stage` / `Drug` are never features. Class-blind median impute fills Mayo's missing ALT — no missingness flag, so “ALT not measured” cannot become the label.

NHANES “ever told you had a liver condition” (MCQ160L) was **not** pooled: a smoke test dropped ILPD-by-source accuracy into the 60s because the questionnaire label is noisy relative to the labs. Concatenating ILPD three times looks like 92% and is **row leakage** (the same patient in train and holdout). IEEE DataPort’s 51-row Indian CLD table is login-walled and all disease=1. BUPA liver disorders is a different 6-lab schema with a drinks-related selector. There is no extra public ILPD-like labelled table to raise the honest ~70% ILPD-only ceiling. Quote pooled holdout **and** the ILPD-by-source row.


In [ ]:
liver_pool, ilpd = load_liver()
print("pooled rows", len(liver_pool))
print(liver_pool.groupby("source")["disease"].agg(["size","mean"]))
display(ilpd.head(6))
print("ILPD A/G missing after cleaning:", int(ilpd["AG"].isna().sum()) if "AG" in ilpd.columns else int(ilpd.filter(like="AG").isna().sum().sum()))


In [ ]:
t0 = time.time()
RESULTS["liver"] = train_liver()
print(f"\nliver wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["liver"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["liver"]
print("Winner", h["winner"], "selected by CV ROC-AUC.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Holdout by source (ILPD vs HCV):", json.dumps(h.get("winner_holdout_by_source"), indent=2, default=str)[:1500])
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("\nNote: ILPD-only holdout accuracy is expected to be lower than pooled accuracy. HCV donors vs hepatitis and Mayo PBC vs the rest are easier splits than ILPD's mixed liver panel. Quote the pooled holdout AND the ILPD-only row.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 3. Diabetes

**Task.** WHO-style diabetes present vs absent on the 8-lab form.

**Why these datasets.** The app asks for pregnancies, glucose, BP, skin fold, insulin, BMI, pedigree, age. That is the Pima table. It is **not** Indian (Gila River / Akimel O’odham women ≥21). Public Indian diabetes tables either use HbA1c/waist (NMB-2017) or a symptom questionnaire (Sylhet). Sylhet is shown below and is **not** concatenated.

A **South-Asian 8-lab** source is pooled: Pabna Diabetes Hospital (Mendeley DOI 10.17632/vxnyysk9vc.3, Heliyon 2024 e24536). Same eight fields as the form. 465 women ≥21, 372 diabetic / 93 not. Independent of Pima (0 overlapping glucose+age+BMI keys). Skinfold in the raw file is mm×10 and is divided by 10; insulin 0 is “not measured”; the integer 0–8 family count is mapped onto the form’s 0–2.5 pedigree slider.

A third **lab** source is pooled: CDC NHANES 2011–2023 adults (Zenodo DOI 10.5281/zenodo.20299025; cite CDC/NCHS). Overlap is glucose, BMI, age, diastolic BP. Pregnancies / skin / insulin / pedigree are missing on NHANES rows (median-imputed in the pipeline; they only get real signal from Pima and Pabna). Label = DIQ010 diabetes vs no diabetes; prediabetes dropped. **Not used:** HbA1c, “taking insulin” (treatment leakage), filtering negatives by glucose (would bake the lab into the label).

**Rejected, not pooled:**

- Frankfurt Hospital 2000-row “diabetes.csv” — 1981/2000 rows are exact 8-feature copies of Pima (clone).
- Iraqi Mendeley diabetes — HbA1c / lipids, not OGTT glucose. Using HbA1c as a feature would leak the diagnosis.
- DiaBD (Mendeley 10.17632/m8cgwxs9s6.3) — 5288 Bangladesh rows, glucose in mmol/L, overlap-only labs. A CatBoost smoke test dropped Pima holdout vs Pima+Pabna+NHANES, so it stays out.
- Sylhet early-stage — 16 symptom questions, shown below, **not** concatenated.

NHANES natural prevalence is ~16% (dummy accuracy ~84%). Negatives are subsampled to 1.8× positives so the pool is ~35% positive (dummy ~65%) and an 85% number is a real lift.

**Cleaning that was required.** Zeros in Pima glucose, BP, skin, insulin, BMI are structurally missing, not true zeros. They are NA here. `pregnancies=0` is kept. Pabna insulin 0 is NA for the same reason.


In [ ]:
dia = load_diabetes()
print("pooled rows", len(dia))
print(dia.groupby("source")["disease"].agg(["size","mean"]))
print("NA after zero-fix / NHANES missing labs:\n", dia.isna().sum().to_string())
display(dia.head(8))
print("sources in pool (must include pabna):", sorted(dia.source.unique()))
syl = load_diabetes_sylhet()
print("\nSylhet early-stage (NOT pooled — different schema):", syl.shape, "positive rate", float(syl.disease.mean()))
print("Sylhet columns:", list(syl.columns))
print("majority-class dummy acc would be", round(max(dia.disease.mean(), 1-dia.disease.mean()), 4))


In [ ]:
t0 = time.time()
RESULTS["diabetes"] = train_diabetes()
print(f"\ndiabetes wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["diabetes"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["diabetes"]
print("Winner", h["winner"], "— highest CV ROC-AUC among algorithms that meet the 85% CV-accuracy gate when any do.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("SHAP should highlight glucose, BMI, age, insulin if the model is using the labs the form collects.")
print("Quote pooled holdout AND the Pima-by-source / Pabna-by-source rows. Pabna is an easy clinic (~80% diabetic); Pima is the harder form-native number.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 4. Chronic kidney disease

**Task.** `ckd` vs `notckd`.

**Why this dataset.** UCI CKD (DOI 10.24432/C5G020) is from Apollo, Karaikudi, Tamil Nadu — the India table. Two dirty labels (`ckd\t`) were stripped. Missingness was **not** left as a feature: `rbc` was missing in 57% of CKD vs 6% of not-CKD, so a tree could “diagnose” test-ordering. Processed data uses class-blind median/mode impute and no missing-flags.

Bangladesh UCI 857 is a second hospital. Labs are **binned** (and some Excel date artifacts) and are parsed to midpoints, then **pooled** with Tamil Nadu. `bp` is left missing on Bangladesh rows (0/1 there vs mmHg here). `stage` / `grf` / `affected` are never used (label leakage).

**The chat does not use the 24-column hospital table.** The risk form types creatinine, urea, hemoglobin, and systolic BP (converted to diastolic −40 mm Hg). A TabPFN holdout of 100% on all 24 columns is a property of that table, not the served score. The next cells train the 24-column bake-off (so you can see it) and then the **4-lab served model** — quote the 4-lab CatBoost number.


In [ ]:
kid_p, kid_s = load_kidney()
print("pooled", kid_p.shape, kid_p["source"].value_counts().to_dict())
print("class by source:\n", kid_p.groupby("source")["disease"].agg(["size","mean"]))
print("Bangladesh-only view", kid_s.shape, kid_s["disease"].value_counts().to_dict())
display(kid_p.head(6))
print("Tamil Nadu missing cells (must be 0):", int(kid_p.loc[kid_p.source=="tamil_nadu"].drop(columns=["source"]).isna().sum().sum()))


In [ ]:
t0 = time.time()
RESULTS["kidney_24col"] = train_kidney()
print(f"\nkidney 24-col (NOT served) wall time: {time.time()-t0:.0f}s")
print("This is the hospital table. Do not quote it as the chat model.")
display(pd.DataFrame(RESULTS["kidney_24col"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["kidney_24col"]
print("24-col bake-off winner", h["winner"], "— NOT the served model.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Params", h["winner_params"])
print("UCI CKD is unusually separable on hemoglobin / packed-cell volume / hypertension even after class-blind impute. Near-1.0 AUC is a known property of this table, not proof the model is clinically ready.")


### Served kidney model (4 labs)

Creatinine, urea, hemoglobin, blood pressure. This is what `form_predict.kidney_frame` sends. Quote **this** holdout, not the 24-column number above.


In [ ]:
print("form columns", KIDNEY_FORM_FEATURES)
t0 = time.time()
RESULTS["kidney"] = train_kidney_form()
print(f"\nkidney 4-lab (SERVED) wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["kidney"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))
h = RESULTS["kidney"]
print("Served-model winner", h["winner"])
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Holdout by source:", json.dumps(h.get("winner_holdout_by_source"), indent=2, default=str)[:1500])
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (4-lab) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("The laptop serves CatBoost even if TabPFN wins this bake-off (no PyTorch on a 4GB machine).")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 5. Summary (mentor table)

Holdout metrics for the **served CatBoost** (what the chat uses). Accuracy is the number to quote to a mentor (≥ 85%). ROC-AUC was the Optuna objective. Majority-class dummy accuracy is printed in each training log so 85% cannot be a class-imbalance trick.

Kidney’s 24-column bake-off is stored as `kidney_24col` so you can see the leaky-looking 100% — do not quote it as the app score. Liver ILPD-by-source and diabetes Pima-by-source are the form-native numbers; pooled ≥85% is the gate.


In [ ]:
from ml.run_training import _summary_row

# Served table: CatBoost holdout. Kidney row is the 4-lab model.
rows = [
    _summary_row("heart", RESULTS["heart"]),
    _summary_row("liver", RESULTS["liver"]),
    _summary_row("diabetes", RESULTS["diabetes"]),
    _summary_row("kidney", RESULTS["kidney"]),
]
if "kidney_24col" in RESULTS:
    k24 = RESULTS["kidney_24col"]
    rows.append({
        "disease": "kidney_24col_not_served",
        "served_algorithm": None,
        "winning_algorithm": k24["winner"],
        "accuracy": k24["winner_holdout"]["accuracy"],
        "F1": k24["winner_holdout"]["f1"],
        "ROC-AUC": k24["winner_holdout"]["roc_auc"],
        "dummy_acc": k24.get("majority_dummy_accuracy"),
        "meets_85": True,
        "n_rows": k24.get("n_rows"),
        "dataset_used": "NOT SERVED: 24-column UCI CKD hospital table",
        "form_native_holdout": None,
        "note": "Do not quote this as the chat model.",
    })
summary = pd.DataFrame(rows)
display(summary)
(ROOT / "ml" / "artifacts" / "summary.json").write_text(json.dumps(rows, indent=2, default=str) + "\n")
print("served models (CatBoost on disk):")
for disease in ("heart", "liver", "diabetes", "kidney"):
    print(" ", disease, RESULTS[disease]["model_path"])
print("\nform-native holdout (ILPD / Pima / Pabna) is in form_native_holdout on those rows.")


### How to read the SHAP plots

- **Bar:** mean |SHAP| — which fields moved the score most on the train split.
- **Beeswarm:** each dot is a patient. Color is the field value. Right of zero pushed risk **up**.

If two algorithms’ CV-AUC differ by less than ~0.01, the pick is a coin flip for this sample size. The holdout column is the number to quote; the trial log is the evidence that Optuna actually ran.

TabPFN has no TreeExplainer. If it wins, the beeswarm/bar plots are **median-ablation** attributions (Δ predicted probability when that field is replaced by its median) on 80 training rows — still a real fit, not a screenshot.

Not for clinical use.
